In [ ]:
!pip install torch
!pip install torchvision

: 

In [ ]:
import torch 
from torch import nn

import torchvision 
from torchvision import datasets
from torchvision import transforms
from torchvision.transforms import ToTensor 

import matplotlib.pyplot as plt

print(torch.__version__)
print(torchvision.__version__)

: 

### Getting Dataset   

In [ ]:
train_data = datasets.FashionMNIST(root='data',
                      train=True,
                      transform=ToTensor(),
                      download=False,
                      target_transform=None)


In [ ]:
test_data = datasets.FashionMNIST(root='data',
                                download=False,
                                train=False,
                                transform=ToTensor(),
                                target_transform=None)


In [ ]:
train_data, test_data

In [ ]:
from torch.utils.data import Dataset, DataLoader 

BATCH_SIZE = 32

train_dataloader = DataLoader(train_data, 
                              batch_size=BATCH_SIZE,
                              shuffle=True,
                              num_workers=4)

test_dataloader = DataLoader(test_data,
                             batch_size=BATCH_SIZE,
                             shuffle=False,
                             num_workers=4)

train_dataloader, test_dataloader

In [ ]:
# Data Attributes
print(f'Length of train_dataloader: {len(train_dataloader)} batches of {BATCH_SIZE}')
print(f'Length of test_dataloader: {len(test_dataloader)} batches of {BATCH_SIZE}')

In [ ]:
train_batch, train_labels_batch = next(iter(train_dataloader))
train_batch.shape, train_labels_batch.shape

In [ ]:
test_batch, test_labels_batch = next(iter(test_dataloader))
test_batch.shape, test_labels_batch.shape

In [ ]:
train_batch[0]

In [ ]:
# image, label = train_loader[0]
# image, label
train_data, test_data

In [ ]:
class_to_idx = train_data.class_to_idx
class_to_idx

In [ ]:
classes = train_data.classes
classes

In [ ]:
train_data.targets

In [ ]:
image, label = train_data[0]

# plt.figure(figsize=(12,7))
print(f'image shape: {image.shape}')
plt.imshow(image.squeeze())
plt.title(classes[label])

In [ ]:
plt.imshow(image.squeeze(), cmap="gray")
plt.title(classes[label])

In [ ]:
# torch.manual_seed(24)
fig = plt.figure(figsize=(12,8))
rows, cols = 4, 4
for i in range(1, rows*cols+1):
    random_idx = torch.randint(0, len(train_data), size=[1]).item()
    img, label = train_data[random_idx]
    fig.add_subplot(rows, cols, i)
    plt.imshow(img.squeeze(), cmap="gray")
    plt.title(classes[label])
    plt.axis(False)

In [ ]:
random_idx = torch.randint(0, len(train_batch), size=[1]).item()
img, label = train_batch[random_idx], train_labels_batch[random_idx]
plt.imshow(img.squeeze(), cmap="gray")
plt.title(classes[label])
plt.axis(False)
print(f'Label: {label}({classes[label]})')

### Baseline Model

In [ ]:
import torch
from  torch import nn
import helper_functions 
from helper_functions import accuracy_fn, print_train_time
from timeit import default_timer as timer


In [ ]:
flatten = nn.Flatten()

output = flatten(train_batch[0])

print(f'Data before flattening: {train_batch[0].shape}')
print(f'Data after flattening: {output.shape}')


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
class FashionMNISTV0(nn.Module):
    def __init__(self, input_shape, hidden_units, output_shape):
        super().__init__()
        self.layer_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=input_shape, out_features=hidden_units),
            nn.ReLU(),
            nn.Linear(in_features=hidden_units, out_features=hidden_units),
            nn.ReLU(),
            nn.Linear(in_features=hidden_units, out_features=output_shape)
        )
    
    def forward(self, x):
        return self.layer_stack(x)


In [ ]:
class FashionMnistV1(nn.Module):
    def __init__(self, input_shape, hidden_units, output_shape):
        super().__init__()
        self.features = nn.Sequential(
            ## Block 1
            nn.Conv2d(in_channels=input_shape, out_channels=hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),  
            nn.MaxPool2d(kernel_size=2),   # channel x 28 x 28 -> hidden_units x 14 x 14

            ## Block 2
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),   # hidden_units x 14 x 14 -> hidden_units x 7 x 7
            
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(hidden_units * 7 * 7, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, output_shape),
        )


    def feature(self, x: torch.Tensor):     
            return self.features(x)
    

    def forward(self, x: torch.Tensor):
        return self.classifier(self.features(x))

In [ ]:
torch.manual_seed(42)

model_0 = FashionMNISTV0(input_shape=784, # 28 * 28
                         hidden_units=8,
                         output_shape=len(classes)).to(device)
  
model_0

model_1 = FashionMnistV1(input_shape=1, hidden_units=8, output_shape=len(classes)).to(device)

next(iter(model_0.parameters())), next(iter(model_1.parameters()))

In [ ]:
feature = FashionMnistV1(input_shape=1, hidden_units=8, output_shape=10).feature(train_batch[1])

print(f'Train Batch dimension(shape): {train_batch.shape}')
print(f'Feature shape: {feature.shape}')                                  
# print(f'Feature values: {feature[:]}')

In [ ]:
x = train_batch[:2].to(device) # Keeps batch dimension: [1, 1, 28, 28]
print(f'Input shape: {x.shape}')

print(model_1(x))

In [ ]:
model_0.state_dict()

#### Setting up Loss, Optimizer, and Evaluation Metrics

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model_0.parameters(),
                            lr=0.01,
                            momentum=0.9)



#### Model Training

In [ ]:
from tqdm.auto import tqdm

def train_model(model: nn.Module,
                train_data: torch.Tensor,
                test_data: torch.Tensor,
                loss_fn,
                optimizer,
                Epochs, 
                device=device):
    """ Model Training Loop"""

    torch.manual_seed(42)
    torch.cuda.manual_seed(42)

    train_start_time = timer()

    train_loss = 0
    for epoch in tqdm(range(Epochs)):
        for batch, (X, y) in enumerate(train_data):
            model.train()

            X, y = X.to(device), y.to(device)
            
            # 1. Forward Pass
            y_logits = model(X)
            y_pred = torch.softmax(y_logits, dim=1).argmax(dim=1)

            # 2. Calculate Loss
            loss = loss_fn(y_logits, y)
            train_loss += loss

            accuracy = accuracy_fn(y, y_pred)

            # 3. Zero_grad
            optimizer.zero_grad()

            # 4. Backpropagation
            loss.backward()

            # 5. Optimizer step
            optimizer.step()

        model.eval()
        
        
        if epoch % 10 == 0:
            for batch, (X, y) in enumerate(test_data):
                test_logits = model(X)
                test_preds = torch.softmax(test_logits, dim=1).argmax(dim=1)
                test_loss = loss_fn(test_logits, y)
                test_accuracy = accuracy_fn(y, test_preds)

            print(f'Epoch: {epoch} | Train Loss: {loss} | Accuracy: {accuracy}% | Test Loss: {test_loss} | Test Accuracy: {test_accuracy}%')
    
    train_loss  /= len(train_data)

    train_end_time = timer()

    print_train_time(train_start_time, train_end_time, device=str(next(iter(model.parameters()))))

In [ ]:
Epochs = 20

train_model(model_0, train_data=train_dataloader, test_data=test_dataloader, loss_fn=loss_fn, optimizer=optimizer, Epochs=Epochs)

In [ ]:
optimizer_1 = torch.optim.Adam(model_1.parameters(), lr=0.01)

In [ ]:
Epochs = 20
train_model(model_1, train_data=train_dataloader, test_data=test_dataloader, loss_fn=loss_fn, optimizer=optimizer_1, Epochs=Epochs)

In [ ]:
def eval_model(model: nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               accuracy_fn):
    
    loss, acc = 0, 0
    model.eval()
    with torch.inference_mode():
        for X, y in data_loader:
            y_logits = model(X)
            y_preds = torch.softmax(y_logits, dim=1).argmax(dim=1) 
            # Calculating Loss
            loss += loss_fn(y_logits, y)
            acc = accuracy_fn(y_true=y, y_pred=y_preds)
    
        loss /= len(data_loader)
        acc /= len(data_loader)

    return {"model_name": model.__class__.__name__,
            "model_loss": loss.item(),
            "model_acc": acc}



In [ ]:
model_results = eval_model(model_1, test_dataloader, loss_fn=loss_fn, accuracy_fn=accuracy_fn)
model_results

### Saving Model

In [ ]:
from pathlib import Path

MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)
MODEL_NAME = "01_pytorch_computer_vision_model_1.pth"

MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

print(f'Saving model to: {MODEL_SAVE_PATH}')
torch.save(obj=model_1.state_dict(), f=MODEL_SAVE_PATH)

## Loading Model

In [ ]:
load_model = FashionMnistV1(input_shape=1, hidden_units=8, output_shape=10).to(device)

load_model.load_state_dict(torch.load(MODEL_SAVE_PATH))

eval_model(load_model, test_dataloader, loss_fn=loss_fn, accuracy_fn=accuracy_fn)